# Fine-tune AraT5v2 on Egyptian Franco↔Arabic

**Project:** [github.com/MoazReda/franco](https://github.com/MoazReda/franco)

Trains a single bidirectional Seq2Seq model that handles both directions via prefix tokens:
- `<2ar>` — Franco → Arabic
- `<2franco>` — Arabic → Franco

## How to use this notebook on Kaggle

1. **Add accelerator**: Settings → Accelerator → GPU T4 x2 (or P100).
2. **Add dataset**: upload the three CSVs (`train_augmented.csv`, `val.csv`, `test.csv`) as a private Kaggle dataset named `franco-translator-data`.
3. (Optional) Add a Kaggle Secret `HF_TOKEN` with a HuggingFace write token if you want to push the trained model to the Hub at the end.
4. Run All.

## Baseline numbers (must-beat floor)

| Direction | BLEU | chrF |
|-----------|------|------|
| Franco → Arabic | 3.94 | 33.90 |
| Arabic → Franco | 15.38 | 42.52 |

## 1. Install / verify dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece sacrebleu

In [ ]:
import torch
import transformers
import datasets

print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('datasets     :', datasets.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))

## 2. Config

All knobs live in one cell so changing them is a single edit. If you're not running on Kaggle, just point `DATA_DIR` at your `data/splits/` folder.

In [ ]:
from pathlib import Path

# Kaggle dataset path. Adjust if your dataset is named differently.
DATA_DIR = Path('/kaggle/input/franco-translator-data')
OUTPUT_DIR = Path('/kaggle/working/franco-translator-v1')

MODEL_NAME = 'UBC-NLP/AraT5v2-base-1024'
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

NUM_EPOCHS = 5
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WARMUP_RATIO = 0.1
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 0.01

SEED = 42
FP16 = torch.cuda.is_available()

PREFIX_TO_AR = '<2ar>'
PREFIX_TO_FRANCO = '<2franco>'

# Optional: push the trained model to the HuggingFace Hub at the end.
PUSH_TO_HUB = False
HUB_REPO_ID = 'MoazReda/franco-translator-v1'

## 3. Load and reshape the data

For each `(franco, arabic)` row we emit two training examples — one in each direction — by prepending a prefix token to the source side.

In [ ]:
import pandas as pd


def build_bidirectional(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        franco = str(row['franco']).strip()
        arabic = str(row['arabic']).strip()
        if not franco or not arabic:
            continue
        rows.append({'input_text': f'{PREFIX_TO_AR} {franco}', 'target_text': arabic})
        rows.append({'input_text': f'{PREFIX_TO_FRANCO} {arabic}', 'target_text': franco})
    return pd.DataFrame(rows)


train_raw = pd.read_csv(DATA_DIR / 'train_augmented.csv', encoding='utf-8-sig')
val_raw = pd.read_csv(DATA_DIR / 'val.csv', encoding='utf-8-sig')
test_raw = pd.read_csv(DATA_DIR / 'test.csv', encoding='utf-8-sig')

train_df = build_bidirectional(train_raw)
val_df = build_bidirectional(val_raw)
test_df = build_bidirectional(test_raw)

print(f'train: {len(train_df)} pairs (from {len(train_raw)} source rows)')
print(f'val  : {len(val_df)} pairs (from {len(val_raw)} source rows)')
print(f'test : {len(test_df)} pairs (from {len(test_raw)} source rows)')
train_df.head(4)

## 4. Tokenizer + model

We add the two prefix tokens as special tokens and resize the model's embedding table so the new vocab entries actually get trained.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
added = tokenizer.add_special_tokens({'additional_special_tokens': [PREFIX_TO_AR, PREFIX_TO_FRANCO]})
print(f'Added {added} prefix tokens to vocab. New size: {len(tokenizer)}')

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if added > 0:
    model.resize_token_embeddings(len(tokenizer))

n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params / 1e6:.1f}M')

## 5. Tokenize the datasets

In [ ]:
from datasets import Dataset


def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


def to_hf(df: pd.DataFrame):
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(tokenize, batched=True, remove_columns=ds.column_names)


train_ds = to_hf(train_df)
val_ds = to_hf(val_df)
test_ds = to_hf(test_df)
print(train_ds)

## 6. Metrics — BLEU + chrF

`metric_for_best_model='chrf'` because BLEU is unstable on a 90-example test set; chrF correlates better with quality on small evaluations.

In [ ]:
import numpy as np
import sacrebleu


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [p.strip() for p in decoded_preds]
    refs = [[r.strip()] for r in decoded_labels]
    bleu = sacrebleu.corpus_bleu(decoded_preds, list(zip(*refs))).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, list(zip(*refs))).score
    return {'bleu': bleu, 'chrf': chrf}

## 7. Train

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    warmup_ratio=WARMUP_RATIO,
    label_smoothing_factor=LABEL_SMOOTHING,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    logging_steps=50,
    seed=SEED,
    fp16=FP16,
    report_to=[],
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Final evaluation on the held-out test split

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
print('Test metrics (combined directions):')
for k, v in test_metrics.items():
    print(f'  {k:24s} {v:.4f}')

### Per-direction breakdown

Combined metrics blur the asymmetry between the two directions. Split the test set by `direction` (recoverable from the prefix token) and report each separately.

In [ ]:
def split_test_by_direction(df: pd.DataFrame):
    mask = df['input_text'].str.startswith(PREFIX_TO_AR)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)


fa2ar_df, ar2fa_df = split_test_by_direction(test_df)
fa2ar_ds = to_hf(fa2ar_df)
ar2fa_ds = to_hf(ar2fa_df)

fa2ar_metrics = trainer.evaluate(eval_dataset=fa2ar_ds, metric_key_prefix='fa2ar')
ar2fa_metrics = trainer.evaluate(eval_dataset=ar2fa_ds, metric_key_prefix='ar2fa')

print('\n┌─────────────────┬─────────┬─────────┬──────────┐')
print('│ Direction       │ BLEU    │ chrF    │ Δ vs base│')
print('├─────────────────┼─────────┼─────────┼──────────┤')
fa_bleu = fa2ar_metrics['fa2ar_bleu']
fa_chrf = fa2ar_metrics['fa2ar_chrf']
ar_bleu = ar2fa_metrics['ar2fa_bleu']
ar_chrf = ar2fa_metrics['ar2fa_chrf']
print(f'│ Franco → Arabic │ {fa_bleu:6.2f}  │ {fa_chrf:6.2f}  │ {fa_bleu - 3.94:+6.2f}   │')
print(f'│ Arabic → Franco │ {ar_bleu:6.2f}  │ {ar_chrf:6.2f}  │ {ar_bleu - 15.38:+6.2f}   │')
print('└─────────────────┴─────────┴─────────┴──────────┘')

## 9. Sample predictions

In [ ]:
def generate(text: str, prefix: str, num_beams: int = 4) -> str:
    inputs = tokenizer(f'{prefix} {text}', return_tensors='pt', truncation=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=num_beams)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


print('=== Franco → Arabic ===')
for franco, arabic in zip(test_raw['franco'].head(8), test_raw['arabic'].head(8)):
    pred = generate(franco, PREFIX_TO_AR)
    print(f'src : {franco}')
    print(f'ref : {arabic}')
    print(f'pred: {pred}')
    print()

print('=== Arabic → Franco ===')
for arabic, franco in zip(test_raw['arabic'].head(8), test_raw['franco'].head(8)):
    pred = generate(arabic, PREFIX_TO_FRANCO)
    print(f'src : {arabic}')
    print(f'ref : {franco}')
    print(f'pred: {pred}')
    print()

## 10. Save the model

The best checkpoint (by val chrF) is already loaded into `model` because `load_best_model_at_end=True`. Save it as a single artifact you can download from `/kaggle/working/franco-translator-v1` or push to the HuggingFace Hub.

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f'Saved to {OUTPUT_DIR}')
!ls -lh {OUTPUT_DIR}

In [ ]:
# Optional: push to HuggingFace Hub. Requires a Kaggle Secret named HF_TOKEN.
if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)
    print(f'Pushed to https://huggingface.co/{HUB_REPO_ID}')